# Image Model Deployment to Databricks Model Serving

This notebook deploys a registered image model to Databricks Model Serving on GPU_LARGE instances.

## Install Required Packages

In [ ]:
%pip install mlflow[databricks] databricks-sdk --upgrade
dbutils.library.restartPython()

## Setup Widgets for Configuration

In [ ]:
dbutils.widgets.text("catalog", "main", "Catalog Name")
dbutils.widgets.text("schema", "default", "Schema Name")
dbutils.widgets.text("model_name", "image_model", "Model Name")
dbutils.widgets.text("endpoint_name", "image_model_endpoint", "Endpoint Name")
dbutils.widgets.text("model_version", "1", "Model Version")

## Read Configuration from Widgets

In [ ]:
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
model_name = dbutils.widgets.get("model_name")
endpoint_name = dbutils.widgets.get("endpoint_name")
model_version = dbutils.widgets.get("model_version")

registered_model_name = f"{catalog}.{schema}.{model_name}"

print(f"Catalog: {catalog}")
print(f"Schema: {schema}")
print(f"Registered Model: {registered_model_name}")
print(f"Model Version: {model_version}")
print(f"Endpoint Name: {endpoint_name}")

## Initialize Databricks Workspace Client

In [ ]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import (
    EndpointCoreConfigInput,
    ServedEntityInput,
    AutoCaptureConfigInput
)

w = WorkspaceClient()
print(f"Workspace URL: {w.config.host}")

## Deploy Model to Serving Endpoint

This will create or update a model serving endpoint with GPU_LARGE workload size.

In [ ]:
# Check if endpoint exists
try:
    existing_endpoint = w.serving_endpoints.get(endpoint_name)
    print(f"Endpoint '{endpoint_name}' already exists. Updating...")
    endpoint_exists = True
except Exception as e:
    print(f"Endpoint '{endpoint_name}' does not exist. Creating new endpoint...")
    endpoint_exists = False

In [ ]:
# Configure the served entity
served_entity = ServedEntityInput(
    entity_name=registered_model_name,
    entity_version=model_version,
    workload_size="GPU_LARGE",
    scale_to_zero_enabled=True,
    min_provisioned_throughput=0,
    max_provisioned_throughput=0
)

if endpoint_exists:
    # Update existing endpoint
    w.serving_endpoints.update_config_and_wait(
        name=endpoint_name,
        served_entities=[served_entity]
    )
    print(f"Endpoint '{endpoint_name}' updated successfully!")
else:
    # Create new endpoint
    w.serving_endpoints.create_and_wait(
        name=endpoint_name,
        config=EndpointCoreConfigInput(
            served_entities=[served_entity],
            auto_capture_config=AutoCaptureConfigInput(
                catalog_name=catalog,
                schema_name=schema,
                enabled=True
            )
        )
    )
    print(f"Endpoint '{endpoint_name}' created successfully!")

## Get Endpoint Details

In [ ]:
endpoint = w.serving_endpoints.get(endpoint_name)
print(f"\nEndpoint Name: {endpoint.name}")
print(f"Endpoint State: {endpoint.state.ready}")
print(f"Endpoint URL: {w.config.host}/serving-endpoints/{endpoint_name}/invocations")

## Test Endpoint (Optional)

You can test the endpoint with a sample image.

In [ ]:
import base64
import requests
import json

def test_endpoint(image_path):
    """
    Test the deployed endpoint with an image
    
    Args:
        image_path: Path to the image file to test
    """
    # Read and encode image
    with open(image_path, "rb") as f:
        image_data = base64.b64encode(f.read()).decode("utf-8")
    
    # Prepare request
    url = f"{w.config.host}/serving-endpoints/{endpoint_name}/invocations"
    headers = {
        "Authorization": f"Bearer {w.config.token}",
        "Content-Type": "application/json"
    }
    
    payload = {
        "inputs": {"image": image_data}
    }
    
    # Make request
    response = requests.post(url, headers=headers, json=payload)
    
    if response.status_code == 200:
        print("Prediction successful!")
        print(json.dumps(response.json(), indent=2))
    else:
        print(f"Error: {response.status_code}")
        print(response.text)
    
    return response

# Example usage (uncomment and provide image path to test):
# test_endpoint("/path/to/test/image.jpg")